In [1]:
import pandas as pd
import numpy as np
import os
import time

# Analyse intensité carbonique industrielles en Belgique

## Objectif

Construire un outil d'analyse de l'**intensité carbonique** des sites industriels 
belges, à destination des agences régionales de l'environnement (AwAC, VMM, 
Bruxelles Environnement), pour suivre la trajectoire de décarbonation 
sur la période 2007-2024.

## Périmètre d'analyse

L'analyse s'appuie sur deux sources E-PRTR à des grains complémentaires :
- **F1_4 — Émissions par site** : CO2 et autres polluants déclarés au niveau 
  facility (toutes sources confondues : combustion + procédés industriels)
- **F5_2 — Énergie LCP** : consommation par combustible des grandes installations(cheminée) 
  de combustion (≥ 50 MW), unité réglementée par la directive IED

## Indicateur principal

**Intensité carbonique = CO2 émis (t) / énergie consommée (TJ)**

Calculé au niveau site (numérateur F1_4, dénominateur agrégé depuis F5_2).

## Chargement des tables

In [ ]:
f1_path = "./User friendly .csv files/F1_4_Air_Releases_Facilities.csv"
f5_path = "./User friendly .csv files/F5_2_LCP_Energy_Emissions.csv"
f2_4_path = "./User friendly .csv files/F2_4_Water_Releases_Facilities.csv"
f3_2_path = "./User friendly .csv files/F3_2_Transfers_Facilities.csv"
f6_1_path="./User friendly .csv files/F6_1_IED_Installations.csv"

df_f1 = pd.read_csv(f1_path, low_memory=False)
df_f5 = pd.read_csv(f5_path, low_memory=False)
df_f2_4 = pd.read_csv(f2_4_path, low_memory=False)
df_f3_2 = pd.read_csv(f3_2_path, low_memory=False)
df_f6_1=pd.read_csv(f6_1_path,low_memory=False)

print("F1_4 shape :", df_f1.shape)
print("F5_2 shape :", df_f5.shape)
print("F2_4 shape :", df_f2_4.shape)
print("F3_2 shape :", df_f3_2.shape)
print("F6_1 shape :", df_f6_1.shape)

In [ ]:
# Focus Belgique
be_f1 = df_f1[(df_f1["countryName"] == "Belgium") & (df_f1["Pollutant"] == "Carbon dioxide (CO2)")].copy()
be_f5 = df_f5[df_f5["countryName"] == "Belgium"].copy()
be_f2_4 = df_f2_4[df_f2_4["countryName"] == "Belgium"].copy()
be_f3_2 = df_f3_2[df_f3_2["countryName"] == "Belgium"].copy()
be_f6_1=df_f6_1[df_f6_1["CountryName"]=="Belgium"].copy()


print("Belgique - F1 CO2 :", be_f1.shape)
print("Belgique - F5 :", be_f5.shape)
print("Belgique - F2_4 :", be_f2_4.shape)
print("Belgique - F3_2 :", be_f3_2.shape)
print("Belgique - F6_1 :", be_f6_1.shape)

print("\nAnnées F1 CO2 :", sorted(be_f1["reportingYear"].dropna().unique()))
print("Années F5 :", sorted(be_f5["reportingYear"].dropna().unique()))
print("Années F2_4 :", sorted(be_f2_4["reportingYear"].dropna().unique()))
print("Années F3_2 :", sorted(be_f3_2["reportingYear"].dropna().unique()))
print("Année F6_1:", sorted(be_f3_2["reportingYear"].dropna().unique()))


## information des tables

**Liens entre les tables E-PRTR**

````
┌─────────────────────────────────────┐
│ F1_4_Air_Releases_Facilities        │
│ Grain : Site × Année × Polluant     │
│ PK : FacilityInspireId              │
└─────────────────┬───────────────────┘
                  │
                  │ 1 facility ──▶ N installations
                  │
                  ▲ parent_facilityInspireId
┌─────────────────┴───────────────────┐
│ F6_1_IED_Installations              │
│ Grain : Installation × Année        │
│ PK : InstallationInspireId          │
│ FK : parent_facilityInspireId       │
└─────────────────┬───────────────────┘
                  │
                  │ 1 installation ──▶ N LCP (?)
                  │ ⚠️ Clé de jointure à confirmer
                  │
                  ▲ ?
┌─────────────────┴───────────────────┐
│ F5_2_LCP_Energy_Emissions           │
│ Grain : LCP × Année × FeatureType   │
│ PK : LCPInspireId + reportingYear   │
│        + featureType                │
└─────────────────────────────────────┘
````

**Note** : la clé de jointure F6_1 ↔ F5_2 reste à confirmer.
Hypothèse à vérifier : décomposition de `LCPInspireId` permettant
de retrouver le `InstallationInspireId` parent.
````



### F5_2 — Consommation énergétique et émissions des LCP
**Grain : 1 LCP × 1 année × 1 type de mesure (combustible OU polluant OU caractéristique technique)**

In [ ]:
be_f5.info()

In [ ]:
be_f5.head(5)

In [ ]:
be_f5['featureType'].unique()

### F1_4 — Émissions de polluants atmosphériques par site industriel
**Grain : 1 site (facility) × 1 année × 1 polluant**

In [ ]:
be_f1.info()

In [28]:
print(f"nombre de polluants référencés: {len(df_f1['Pollutant'].unique())}")

nombre de polluants référencés: 69


In [ ]:
be_f1.head()

In [ ]:
silver_f1_facility_co2 = (
    be_f1.loc[be_f1["reportingYear"].between(2016, 2024),
              ["FacilityInspireId", "facilityName", "city", "Longitude", "Latitude",
               "EPRTR_SectorCode", "EPRTR_SectorName", "EPRTRAnnexIMainActivity",
               "reportingYear", "Releases"]]
    .rename(columns={"Releases": "facility_co2"})
    .groupby(
        ["FacilityInspireId", "facilityName", "city", "Longitude", "Latitude",
         "EPRTR_SectorCode", "EPRTR_SectorName", "EPRTRAnnexIMainActivity", "reportingYear"],
        as_index=False
    )["facility_co2"].sum()
)

silver_f1_facility_co2.info()


### Table F6_1_IED installation : lien entre table consommation énergétique et emission CO2

In [ ]:
be_f6_1.info()

In [ ]:
be_f6_1.head()

In [ ]:
df_energy_emission.head()

### Données énergie pour la belgique

In [ ]:
# 1 ligne = 1 LCP × 1 année × 1 mesure

print(df_energy_emission[df_energy_emission['countryName']=='Belgium'].shape)

In [ ]:
mask_belgique= df_energy_emission['countryName']=='Belgium'
df_be_energy_emission = df_energy_emission[mask_belgique]

print(f'consommation énergie: {df_be_energy_emission['featureType'].unique()}')
print(f'unités énergie:{df_be_energy_emission['unit'].unique()}')
print(f'rapports disponibles:{df_be_energy_emission['reportingYear'].unique()}')

**Table emission énergie** : une ligne = 1 LCP × 1 année × 1 mesure.

La colonne `featureType` contient trois types de mesures distincts :
- **Combustibles** (NaturalGas, Coal, Lignite, Biomass, LiquidFuels, OtherGases, OtherSolidFuels, Peat) : énergie consommée, exprimée en TJ dans `featureValue`
- **Polluants** (NOX, DUST, SO2) : émissions, exprimées en tonnes
- **Caractéristiques techniques** (LCPCharacteristics) : puissance installée (MW) ou heures de fonctionnement

**Note méthodologique** :
- Pour obtenir l'énergie totale consommée par un LCP sur une année, il faudra **agréger** (somme) les valeurs des `featureType` correspondant aux combustibles.
- Le CO2 n'est **pas déclaré directement** dans la table. Il faudra l'estimer en appliquant des **facteurs d'émission GIEC** propres à chaque combustible (calcul prévu en phase Silver).

In [ ]:
print(df_be_energy_emission.groupby(['featureType', 'unit']).size())

### Nombre de sites industriels en belgique

In [ ]:
mask_CO2=df_emission_facilities['Pollutant']=='Carbon dioxide (CO2)'
mask_be_emission=df_emission_facilities['countryName']=='Belgium'
df_be_emission_CO2=df_emission_facilities[mask_CO2 & mask_be_emission]


# Combien de sites ont des données complètes sur plusieurs années ?
sites_par_annee = df_be_emission_CO2.groupby('facilityName')['reportingYear'].nunique()
print(f"Nombre de site ayant un rapport annuel d'emission: {sites_par_annee.sort_values(ascending=False).shape[0]}")


In [ ]:
# Répartition - combien d'années par site ?
print(sites_par_annee.value_counts().sort_index())

# Les sites les plus complets
print(sites_par_annee.sort_values(ascending=False).shape)

In [ ]:
sites_complets = sites_par_annee[sites_par_annee == 18].index
print(df_emission_facilities[df_emission_facilities['facilityName'].isin(sites_complets)]['facilityName'].unique())

### Données d'emission CO2 au niveau européen

In [ ]:
# Combien de sites par pays ?
print(df_emission_facilities[mask_CO2].groupby('countryName')['facilityName'].nunique().sort_values(ascending=False))

In [ ]:
# Combien d'années couvertes par pays ?
print(df_emission_facilities[mask_CO2].groupby('countryName')['reportingYear'].nunique().sort_values(ascending=False))

In [ ]:
# Inspection des LCPInspireId belges
df_be_lcp = df_energy_emission[df_energy_emission['countryName']=='Belgium']
print(df_be_lcp['LCPInspireId'].head(10).tolist())

# Et comparer avec les InstallationInspireId belges de F6_1
df_be_ied = df_IED_installation[df_IED_installation['CountryName']=='Belgium']
print(df_be_ied['InstallationInspireId'].head(10).tolist())
